# CUDA Optimization Course - Module 6: Complete LLM Optimization

Apply all optimization techniques to a real LLM model and identify bottlenecks.

## Setup and Device Configuration

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.profiler import profile, record_function, ProfilerActivity
from torch.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, TensorDataset
import time
import numpy as np
from typing import Dict, Tuple

# Device detection
if torch.cuda.is_available():
    device = 'cuda'
    print(f"CUDA: {torch.cuda.get_device_name()}")
    print(f"CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = 'mps'
    print("Using MPS (Apple Silicon M3)")
else:
    device = 'cpu'
    print("Using CPU")

print(f"PyTorch: {torch.__version__}")
print(f"Selected device: {device}")

## Build a Small LLM Model

class Attention(nn.Module):
    """Multi-head self-attention layer"""
    def __init__(self, d_model: int, nhead: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        
        # Linear projections
        Q = self.q_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.out_proj(context)
        
        return output

class FeedForward(nn.Module):
    """Position-wise feed-forward network"""
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    """Single transformer block: Attention + FFN"""
    def __init__(self, d_model: int, nhead: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attention = Attention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Attention with pre-norm
        attn_out = self.attention(self.norm1(x))
        x = x + self.dropout(attn_out)
        
        # FFN with pre-norm
        ffn_out = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_out)
        
        return x

class SmallLLM(nn.Module):
    """Small Language Model for optimization demonstration"""
    def __init__(
        self,
        vocab_size: int = 50257,
        d_model: int = 384,
        nhead: int = 6,
        num_layers: int = 6,
        d_ff: int = 1536,
        max_seq_len: int = 1024,
        dropout: float = 0.1
    ):
        super().__init__()
        self.d_model = d_model
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, nhead, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Tie embeddings
        self.lm_head.weight = self.embedding.weight
    
    def forward(self, input_ids, use_checkpoint=False):
        seq_len = input_ids.shape[1]
        pos_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        x = self.embedding(input_ids) + self.pos_embedding(pos_ids)
        
        for layer in self.layers:
            if use_checkpoint:
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        
        x = self.norm(x)
        logits = self.lm_head(x)
        
        return logits

# Create model
model = SmallLLM(
    vocab_size=50257,
    d_model=384,
    nhead=6,
    num_layers=6,
    d_ff=1536
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Parameters: {num_params / 1e6:.1f}M")
print(f"Model Size: {num_params * 4 / 1e9:.2f} GB (FP32)")

## Create Optimized DataLoader

# Create dummy dataset
num_samples = 1000
seq_length = 256

input_ids = torch.randint(0, 50257, (num_samples, seq_length))
labels = torch.randint(0, 50257, (num_samples, seq_length))

dataset = TensorDataset(input_ids, labels)

# Optimized DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4 if device != 'mps' else 0,  # MPS doesn't support multiprocessing well
    pin_memory=device == 'cuda',
    prefetch_factor=2,
    persistent_workers=True
)

print(f"Dataset: {len(dataset)} samples")
print(f"Batch size: 32")
print(f"Batches per epoch: {len(dataloader)}")

## Baseline: Without Optimizations

def train_step(model, dataloader, device, use_amp=False, use_checkpoint=False, num_steps=10):
    """Single training loop"""
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = GradScaler(device=device) if use_amp else None
    
    total_loss = 0
    
    torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
    start = time.perf_counter()
    
    for step, (input_ids, labels) in enumerate(dataloader):
        if step >= num_steps:
            break
        
        input_ids = input_ids.to(device, non_blocking=device == 'cuda')
        labels = labels.to(device, non_blocking=device == 'cuda')
        
        optimizer.zero_grad()
        
        if use_amp:
            with autocast(device_type=device, dtype=torch.float16):
                logits = model(input_ids, use_checkpoint=use_checkpoint)
                loss = F.cross_entropy(logits.view(-1, 50257), labels.view(-1))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(input_ids, use_checkpoint=use_checkpoint)
            loss = F.cross_entropy(logits.view(-1, 50257), labels.view(-1))
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
    
    torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
    elapsed = time.perf_counter() - start
    
    return elapsed, total_loss / num_steps

print("Baseline Training (FP32, no checkpointing):")
baseline_time, baseline_loss = train_step(model, dataloader, device, use_amp=False, use_checkpoint=False, num_steps=10)
print(f"Time: {baseline_time:.2f}s")
print(f"Loss: {baseline_loss:.4f}")
print(f"Throughput: {32 * 10 / baseline_time:.0f} samples/sec")

## Optimization 1: AMP Only

model = SmallLLM().to(device)  # Fresh model

print("\nWith AMP (FP16):")
amp_time, amp_loss = train_step(model, dataloader, device, use_amp=True, use_checkpoint=False, num_steps=10)
print(f"Time: {amp_time:.2f}s")
print(f"Loss: {amp_loss:.4f}")
print(f"Throughput: {32 * 10 / amp_time:.0f} samples/sec")
print(f"Speedup: {baseline_time / amp_time:.2f}x")

## Optimization 2: Gradient Checkpointing Only

model = SmallLLM().to(device)  # Fresh model

print("\nWith Gradient Checkpointing:")
ckpt_time, ckpt_loss = train_step(model, dataloader, device, use_amp=False, use_checkpoint=True, num_steps=10)
print(f"Time: {ckpt_time:.2f}s")
print(f"Loss: {ckpt_loss:.4f}")
print(f"Throughput: {32 * 10 / ckpt_time:.0f} samples/sec")
print(f"vs Baseline: {baseline_time / ckpt_time:.2f}x")

## Optimization 3: Combined (AMP + Checkpointing)

model = SmallLLM().to(device)  # Fresh model

print("\nCombined Optimization (AMP + Checkpointing):")
combined_time, combined_loss = train_step(model, dataloader, device, use_amp=True, use_checkpoint=True, num_steps=10)
print(f"Time: {combined_time:.2f}s")
print(f"Loss: {combined_loss:.4f}")
print(f"Throughput: {32 * 10 / combined_time:.0f} samples/sec")
print(f"Speedup vs Baseline: {baseline_time / combined_time:.2f}x")

## Optimization 4: With torch.compile()

model = SmallLLM().to(device)  # Fresh model

try:
    model_compiled = torch.compile(model, mode='reduce-overhead')
    print("\nWith torch.compile (reduce-overhead mode):")
    
    # Warmup
    for _ in range(3):
        input_ids = torch.randint(0, 50257, (32, 256)).to(device)
        with torch.no_grad():
            _ = model_compiled(input_ids, use_checkpoint=False)
    
    compile_time, compile_loss = train_step(model_compiled, dataloader, device, use_amp=True, use_checkpoint=False, num_steps=10)
    print(f"Time: {compile_time:.2f}s")
    print(f"Throughput: {32 * 10 / compile_time:.0f} samples/sec")
    print(f"Speedup vs Baseline: {baseline_time / compile_time:.2f}x")
except Exception as e:
    print(f"torch.compile not supported on {device}")

## Layer-wise Profiling

model = SmallLLM().to(device)
model.eval()

input_ids = torch.randint(0, 50257, (32, 256)).to(device)

activities = [ProfilerActivity.CPU]
if device == 'cuda':
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
    with_stack=False
) as prof:
    with torch.no_grad():
        logits = model(input_ids)
        loss = logits.sum()

print("\nLayer-wise Profiling (Top operations):")
print(prof.key_averages().table(sort_by="cpu_time_total" if device != 'cuda' else "cuda_time_total", row_limit=20))

## Performance Summary

```
Summary of Optimization Techniques:

Technique                  Speedup    Memory Saved    Use Case
─────────────────────────────────────────────────────────────
Baseline                   1.0x       -               Reference
AMP (FP16)                 1.5-3.0x   ~50%            Large models
Gradient Checkpointing     0.7-0.8x   ~50%            Memory limited
AMP + Checkpointing        1.2-2.5x   ~60%            Very large models
torch.compile()            1.3-2.0x   -               Inference/export
Combined all               2.0-4.0x   ~60%            Maximum optimization

M3 MacBook Specific:
- Use MPS for GPU acceleration
- AMP works best (big memory savings)
- Batch size 32-64 optimal
- num_workers should be 0 (causes issues on Mac)
- Focus on memory efficiency for better thermal performance
```

## Key Takeaways

1. **Layer Profiling**: Identify bottleneck layers (usually attention)
2. **AMP First**: Biggest bang for buck, especially on M3
3. **Checkpointing**: Essential for large models
4. **Combine Techniques**: Multiplicative benefits
5. **DataLoader**: Proper configuration essential
6. **torch.compile(): ** Future-proof optimization
7. **Hardware-Aware**: Tailor optimizations to your device